In [1]:
# 필요한 라이브러리 불러오기
import ee  # Google Earth Engine 파이썬 API
import requests  # 웹 요청 (썸네일 이미지 다운로드에 사용)
import pandas as pd
from PIL import Image  # 이미지 파일 열고 저장
from io import BytesIO  # 이미지 바이트 데이터를 PIL로 읽기 위한 버퍼
from datetime import datetime, timedelta  # 날짜 처리용
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [2]:
SAVE_PAHT = '../data/raw/apt_images/'
DATA_PATH = '../data/interim/apt/apt_with_long_lat.csv'

In [3]:
df = pd.read_csv(DATA_PATH)
df.dropna(inplace=True)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23395 entries, 0 to 23572
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   단지명         23395 non-null  object 
 1   전용면적(㎡)     23395 non-null  float64
 2   층           23395 non-null  int64  
 3   건축년도        23395 non-null  int64  
 4   도로명         23395 non-null  object 
 5   면적당 단가(만원)  23395 non-null  float64
 6   아파트 나이      23395 non-null  int64  
 7   계약일자        23395 non-null  object 
 8   alpha       23395 non-null  float64
 9   경도          23395 non-null  float64
 10  위도          23395 non-null  float64
dtypes: float64(5), int64(3), object(3)
memory usage: 2.1+ MB


In [5]:
df.head()

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,계약일자,alpha,경도,위도
0,중앙하이츠,59.91,10,1998,덕릉로84길 7,6.766156,22,2020-07-11,0.266667,127.076754,37.659925
1,북아현경남,59.77,7,1996,북아현로 40,7.499383,24,2020-07-11,1.000000,126.956170,37.561195
2,강남엘에이치1단지,84.83,6,2013,헌릉로571길 20,7.401580,7,2020-07-11,1.000000,127.102102,37.467098
3,중계센트럴파크,59.75,13,2016,덕릉로70가길 21,7.066081,4,2020-07-11,1.000000,127.059507,37.642869
4,주공17단지,49.94,7,1989,덕릉로66길 17,6.967225,31,2020-07-11,0.000000,127.053952,37.643712


In [6]:



# -----------------------------------------------------------
# 1. Google Earth Engine 초기화
# -----------------------------------------------------------
ee.Authenticate()
ee.Initialize(project='aptprice-464102')

# -----------------------------------------------------------
# 2. 아파트 거래 데이터 정의
# -----------------------------------------------------------
apt_transactions = df[['위도', '경도', '계약일자']].apply(
    lambda row: {
        'lat': row['위도'],
        'lon': row['경도'],
        'date': row['계약일자']
    }, axis=1
).tolist()

def process_transaction(idx, tx):
    try:
        lat = tx['lat']
        lon = tx['lon']
        tx_date = datetime.strptime(tx['date'], "%Y-%m-%d")
        start_date = (tx_date - timedelta(days=45)).strftime('%Y-%m-%d')
        end_date = (tx_date + timedelta(days=45)).strftime('%Y-%m-%d')

        center = ee.Geometry.Point([lon, lat])
        roi = center.buffer(1500).bounds()

        collection = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(center)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
        )

        count = collection.size().getInfo()
        if count == 0:
            return f"[X] 이미지 없음 (index {idx}): 날짜={tx['date']}"

        image = collection.median()

        stats = image.reduceRegion(
            reducer=ee.Reducer.percentile([2, 98]),
            geometry=roi,
            scale=10,
            maxPixels=1e8
        ).getInfo()

        if not stats:
            return f"[X] 통계 없음 (index {idx}): 날짜={tx['date']}"

        b4_min = stats.get('B4_p2', 500)
        b4_max = stats.get('B4_p98', 3500)
        b3_min = stats.get('B3_p2', 500)
        b3_max = stats.get('B3_p98', 3500)
        b2_min = stats.get('B2_p2', 500)
        b2_max = stats.get('B2_p98', 3500)

        url = image.getThumbURL({
            'region': roi,
            'format': 'jpg',
            'bands': ['B4', 'B3', 'B2'],
            'min': [b4_min, b3_min, b2_min],
            'max': [b4_max, b3_max, b2_max],
            'scale': 10
        })

        if not url or not url.startswith("https://"):
            return f"[X] URL 생성 실패 (index {idx})"

        response = requests.get(url)
        if response.status_code != 200:
            return f"[X] 이미지 요청 실패 (index {idx}): 상태코드 {response.status_code}"

        img = Image.open(BytesIO(response.content))
        img.save(f"{SAVE_PAHT}apt_image_{idx}.jpg")
        return f"[✓] 이미지 저장 완료: apt_image_{idx}.jpg"

    except Exception as e:
        return f"[X] 예외 발생 (index {idx}): {e}"


In [6]:

# 병렬 처리 실행
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_transaction, idx, tx) for idx, tx in enumerate(apt_transactions)]
    for future in tqdm(as_completed(futures), total=len(futures)):
        print(future.result())

  0%|                                      | 1/23395 [00:04<27:32:14,  4.24s/it]

[✓] 이미지 저장 완료: apt_image_3.jpg
[✓] 이미지 저장 완료: apt_image_1.jpg
[✓] 이미지 저장 완료: apt_image_2.jpg


  0%|                                       | 4/23395 [00:04<5:37:50,  1.15it/s]

[✓] 이미지 저장 완료: apt_image_0.jpg


  0%|                                       | 5/23395 [00:05<5:36:55,  1.16it/s]

...

  0%|                                      | 16/23395 [00:18<6:06:35,  1.06it/s]

[✓] 이미지 저장 완료: apt_image_17.jpg


  0%|                                      | 16/23395 [00:19<8:03:43,  1.24s/it]


KeyboardInterrupt: 


KeyboardInterrupt

